In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Okhla Phase-2, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,215.66,311.79,35.62,46.63,54.16,61.57,6.72,0.85,19.97,2.56,15.24,77.13,0.33,191.49,991.00,14.56,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,213.92,307.38,35.86,45.43,53.32,74.38,9.30,1.73,23.70,2.76,13.63,78.79,0.35,241.26,991.00,14.61,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,332.83,478.79,123.43,67.52,136.28,85.38,11.12,2.07,14.31,5.38,35.34,80.70,0.32,258.01,991.00,15.74,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,293.77,387.12,71.86,45.82,82.28,82.49,6.99,1.42,20.23,5.76,39.62,84.55,0.33,183.84,991.00,15.10,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,191.96,277.83,24.04,29.45,35.09,63.16,7.53,1.44,18.25,2.98,7.74,83.49,0.37,117.18,991.00,14.57,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,339.51,524.49,67.76,111.56,115.21,44.24,17.71,2.33,53.30,2.64,16.88,54.85,0.62,63.02,977.32,18.57,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,288.38,454.25,28.83,111.73,82.85,44.73,18.88,1.83,55.49,2.06,12.78,55.18,0.52,98.61,977.43,18.58,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,213.36,375.13,22.80,114.51,79.46,41.43,16.30,1.74,51.35,1.87,10.16,54.56,0.49,112.86,977.43,18.42,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,243.92,399.50,38.27,111.48,90.74,42.65,16.00,1.78,54.99,2.04,13.19,54.93,0.51,87.38,977.23,18.33,0.0,0.0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 20)
          From Date           To Date   PM2.5    PM10     NO    NO2     NOx  \
0  01-01-2025 00:00  02-01-2025 00:00   54.81  311.79  35.62  46.63   54.16   
1  02-01-2025 00:00  03-01-2025 00:00   54.81  307.38  35.86  45.43   53.32   
2  03-01-2025 00:00  04-01-2025 00:00   54.81  478.79  15.71  67.52  136.28   
3  04-01-2025 00:00  05-01-2025 00:00   54.81  387.12  71.86  45.82   82.28   
4  05-01-2025 00:00  06-01-2025 00:00  191.96  277.83  24.04  29.45   35.09   

      NH3    SO2    CO  Ozone  Benzene  Toluene     RH    WS      WD     BP  \
0  61.570   6.72  0.85  19.97    2.560    15.24  77.13  0.33  191.49  991.0   
1  35.395   9.30  1.73  23.70    2.760    13.63  78.79  0.35  241.26  991.0   
2  35.395  11.12  2.07  14.31    1.025     8.92  80.70  0.32  258.01  991.0   
3  35.395   6.99  1.42  20.23    1.025     8.92  84.55  0.33  183.84  991.0   
4  63.160   7.53  1.44  18.25    2.980     7.74  83.49  0.37  117.18  991.0   

      AT   RF  TOT-RF  
0  

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,-0.165820,1.241676,0.874758,-0.057714,0.258931,2.417879,-0.571062,-1.231378,-1.267298,1.461978,0.725286,0.989387,-1.258697,0.333505,1.724420,-2.267589,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,-0.165820,1.195613,0.888377,-0.108852,0.231690,-0.127561,-0.048126,0.716143,-1.121418,1.680265,0.481945,1.104968,-1.179198,1.062299,1.724420,-2.258366,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.165820,2.985986,-0.255056,0.832513,2.922058,-0.127561,0.320767,1.468594,-1.488659,-0.213376,-0.229941,1.237957,-1.298447,1.307573,1.724420,-2.049923,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.165820,2.028496,2.931234,-0.092232,1.170854,-0.127561,-0.516336,0.030084,-1.257129,-0.213376,-0.229941,1.506023,-1.258697,0.221484,1.724420,-2.167979,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,3.165978,0.886964,0.217639,-0.789839,-0.359504,2.572502,-0.406884,0.074346,-1.334567,1.920381,-0.408291,1.432217,-1.099699,-0.754633,1.724420,-2.265745,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.165820,-0.130063,2.698575,2.709276,2.238764,0.732589,1.656484,2.043998,0.036231,1.549293,0.973162,-0.561913,-0.105958,-1.547711,-0.736155,-1.527894,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.165820,2.729667,0.489452,2.716521,1.189339,0.780240,1.893629,0.937452,0.121881,0.916260,0.353473,-0.538936,-0.503454,-1.026558,-0.716369,-1.526049,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.165820,1.903260,0.147274,2.834990,1.079402,0.459325,1.370693,0.738274,-0.040033,0.708887,-0.042523,-0.582105,-0.622703,-0.817892,-0.716369,-1.555563,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.165820,2.157804,1.025135,2.705867,1.445209,0.577966,1.309887,0.826797,0.102326,0.894431,0.415442,-0.556343,-0.543204,-1.191002,-0.752343,-1.572165,0.0,0.0


In [10]:
df.to_excel('Okhla2025.xlsx', index=False)